# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: Engagement and Visibility Move Together (Page 10)**
*   **The Finding:** Pages with high scroll and high engagement score +11.2 health points.
*   **Methodology Question:** The paper uses `Health Score` as the outcome metric, but the methodology appendix (Page 36) states that Health Score is calculated using `scroll depth (20 pts)`. If we use 'high scroll' to define our cohort, and then evaluate that cohort using a score that mathematically includes 'scroll depth', aren't we seeing label leakage? 

**Finding 2: The Content Performance Curve (Page 7)**
*   **The Finding:** Content peaks at 61-90 days and decays after 270 days.
*   **Methodology Question:** Where does the 'age' label come from, and does a cross-sectional snapshot support a lifecycle claim? If we are looking at all pages today, the 365+ day bucket only contains pages that *survived* a whole year without being deleted or unindexed (survivorship bias). To confidently claim that an individual page decays over time, wouldn't a longitudinal validation design—tracking the exact same cohort of URLs over 12 months—be a more honest test than a single-day snapshot of different pages?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold

# 1. Load data
df = pd.read_csv('../outputs/baseline_action_score.csv')
df['pos_change'] = df['pos_second_half'] - df['pos_first_half']
features = ['imp_past15', 'pos_first_half', 'pos_second_half', 'pos_change']
X = df[features]
y = df['dropped_traffic_next15d']
groups = df['client_hash_id']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 2. Naive Split (Random KFold)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
naive_preds = np.zeros(len(df))
for train_idx, val_idx in kf.split(X):
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    naive_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]

# 3. Honest Split (GroupKFold by Client)
gkf = GroupKFold(n_splits=5)
honest_preds = np.zeros(len(df))
for train_idx, val_idx in gkf.split(X, y, groups):
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    honest_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]

# 4. Compare Results
results = []
for k in [20, 50, 100]:
    results.append({
        'K': k, 
        'Naive P@K (Leaky)': precision_at_k(naive_preds, y, k), 
        'Honest P@K (Grouped)': precision_at_k(honest_preds, y, k)
    })

display(pd.DataFrame(results))

,K,Naive P@K (Leaky),Honest P@K (Grouped)
0,20,0.90,0.60
1,50,0.76,0.52
2,100,0.76,0.54


**Observation:** The naive random split dramatically overestimates model performance. By randomly shuffling rows, the model sees pages from the same client domains in both training and testing. It memorizes client-specific absolute positions (like "position 40 is bad for Client A"). When forced to predict on entirely unseen clients in the Grouped split, the precision collapses. The Grouped split is the honest benchmark.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Audit 1: Label-derived features?**
*   Our target is `dropped_traffic_next15d`. 
*   Our features are: `imp_past15`, `pos_first_half`, `pos_second_half`, `pos_change`.
*   **Verdict:** Pass. None of the features are mathematically derived from the target column.

**Audit 2: Future/overlapping windows?**
*   The target window is strictly AFTER the feature window.
*   Features: Report dates between Day -30 and Day -16.
*   Target: Report dates between Day -15 and Day 0.
*   **Verdict:** Pass. The timeline is strictly separated. No overlap.

**Audit 3: Decision-derived features?**
*   We are not using any system flags (like `Fix CTR` or `Zombie Page`) as inputs.
*   **Verdict:** Pass.

**Overall Verdict:** Leakage Audit Passed. Windows are strictly separated and no label-derived features are present.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Unsafe Claim:**
> "Our Random Forest accurately predicts exactly which pages will lose traffic next month, proving that absolute rank positions determine traffic drops."

**Rewritten Safe Claim:**
> "In this observed sample, the model provided directional decision-support for identifying traffic drops. However, measured performance on unseen clients showed that relying on absolute rank positions leads to overfitting, suggesting relative rank changes are a more robust signal."

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.